# 04 · Data Acquisition — PlantDoc (out-of-domain test only)

Downloads PlantDoc (Singh et al., 2020; cropped classification release, CC BY 4.0).
PlantDoc is **never used for training**, so its own train/test folders are pooled.

- **in_scope/**: labels that match a project class (`config.PLANTDOC_CLASS_MAP`),
  saved under the PlantVillage class name. PlantDoc has no healthy maize or healthy
  potato, so 9 of the 11 classes are covered.
- **unseen/**: the other tomato diseases (`config.PLANTDOC_UNSEEN_LABELS`), used to
  test whether the model defers on diseases it was never trained on.

The repository is cloned to Colab's local disk; only the selected images go to Drive.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

## Clone the selected folders to local disk

In [ ]:
from src.data import plantdoc

labels = list(config.PLANTDOC_CLASS_MAP) + config.PLANTDOC_UNSEEN_LABELS
local_repo = plantdoc.clone_sparse(Path("/content/plantdoc_raw"), labels)
print("Cloned", len(labels), "PlantDoc labels to", local_repo)

## Copy to Drive

Starts from an empty PlantDoc folder so a re-run never leaves stale images behind.
Every image Git tracks must arrive on Drive.

In [ ]:
import shutil

import pandas as pd

if config.PLANTDOC_DIR.exists():
    shutil.rmtree(config.PLANTDOC_DIR)

images = pd.DataFrame(
    plantdoc.copy_to_destination(
        local_repo, config.PLANTDOC_CLASS_MAP, config.PLANTDOC_UNSEEN_LABELS, config.PLANTDOC_DIR
    )
)

summary = images.groupby(["subset", "class"]).size().rename("images").reset_index()
display(summary)
print("Total:", len(images), "| in scope:", (images["subset"] == "in_scope").sum(),
      "| unseen:", (images["subset"] == "unseen").sum())

assert images["copied"].all(), images.loc[~images["copied"], "file"].tolist()
missing = set(labels) - set(images["plantdoc_label"])
assert not missing, f"Labels with no images - check spelling in config: {missing}"

## Save the inventory

`plantdoc_images.csv` lists every image with its subset, class, original PlantDoc
label and original PlantDoc split.

In [ ]:
images.drop(columns="copied").to_csv(config.METADATA_DIR / "plantdoc_images.csv", index=False)
print("Saved plantdoc_images.csv to", config.METADATA_DIR)

## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/plantdoc-acquisition",
    message="Acquire PlantDoc images for out-of-domain testing",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))